In [1]:
import sys
import os
sys.path.append("..")

from dotenv import load_dotenv
load_dotenv("../.env")

# Confirm key loaded — prints only first 8 chars, never the full key
print(os.getenv("GROQ_API_KEY")[:8])

gsk_Pj1w


In [2]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [3]:
from langchain_core.messages import HumanMessage

response = llm.invoke([HumanMessage(content="Hello! Can you hear me?")])
print(response.content)

Hello. Yes, I can hear you. Is there something I can help you with or would you like to chat?


In [4]:
from langchain_core.prompts import ChatPromptTemplate

template = """You are a helpful insurance policy assistant. 
Your job is to answer questions about insurance policies accurately.

Use ONLY the following context to answer the question.
If the answer is not found in the context, say "I don't have enough information in the provided policy document to answer this question."
Do NOT make up information or use outside knowledge.
Always mention the page number where you found the answer.

Context:
{context}

Question:
{question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)
print("Prompt template created successfully")

Prompt template created successfully


In [5]:
# See what the prompt looks like when filled in
test_prompt = prompt.invoke({
    "context": "Room rent limit is 1% of sum insured per day. (Page 5)",
    "question": "What is the room rent limit?"
})

print(test_prompt)

messages=[HumanMessage(content='You are a helpful insurance policy assistant. \nYour job is to answer questions about insurance policies accurately.\n\nUse ONLY the following context to answer the question.\nIf the answer is not found in the context, say "I don\'t have enough information in the provided policy document to answer this question."\nDo NOT make up information or use outside knowledge.\nAlways mention the page number where you found the answer.\n\nContext:\nRoom rent limit is 1% of sum insured per day. (Page 5)\n\nQuestion:\nWhat is the room rent limit?\n\nAnswer:', additional_kwargs={}, response_metadata={})]


In [6]:
import sys
import os
sys.path.append("..")

from dotenv import load_dotenv
load_dotenv("../.env")

from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from app.services.vectorstore import get_vectorstore
from app.services.embedder import get_embeddings
from app.prompts.qa_prompt import get_qa_prompt

# Initialize all components
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
vectorstore = get_vectorstore()
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
prompt = get_qa_prompt()
parser = StrOutputParser()

print("All components ready")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

All components ready


In [7]:
def format_docs(docs):
    # Combines all retrieved chunks into one string
    # Adds page number so LLM can cite it
    return "\n\n".join(
        f"[Page {doc.metadata['page']}]\n{doc.page_content}"
        for doc in docs
    )

In [8]:
chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | parser
)

print("Chain built successfully")

Chain built successfully


In [10]:
questions = [
    "What is the waiting period for pre-existing diseases?",
    "What is the room rent limit per day?",
    "Is maternity covered? What is the sub-limit?",
    "What are the exclusions in this policy?",
    "What is the sum insured options available?"
]

for question in questions:
    
    print(f"Q: {question}")
    answer = chain.invoke(question)
    print(f"A: {answer}")
    print(f"\n{'='*60}")

Q: What is the waiting period for pre-existing diseases?
A: The waiting period for pre-existing diseases is 24 months from the date of inception of the policy (Page 11) and also 36 months for coverage under the policy after the expiry for any pre-existing disease, subject to the same being declared at the time of application and accepted by Insurer (Page 10). Additionally, pre-existing diseases are defined as conditions diagnosed within 48 months prior to the effective date of the policy (Page 4).

Q: What is the room rent limit per day?
A: The room rent limit per day is up to 1% of the Sum Insured. I found this information on Page 5 and also on Page 18.

Q: Is maternity covered? What is the sub-limit?
A: Maternity, specifically Delivery Expenses, is covered under this policy (Page 0, B(13), and Page 6, Special Conditions applicable for this section). The sub-limit for Delivery Expenses, including pre-natal and postnatal expenses, is up to 10% of the Sum Insured (Page 0, B(13)).

Q: Wh

In [11]:
from langchain_core.runnables import RunnableParallel

# Enhanced chain that also returns which chunks were used
chain_with_sources = RunnableParallel(
    answer=chain,
    sources=(retriever | format_docs)
)

result = chain_with_sources.invoke("What is the waiting period?")
print(f"Answer:\n{result['answer']}")
print(f"\nSources used:\n{result['sources']}")

Answer:
The waiting period is 24 months from the date of inception of the policy. I found this information on Page 11 and also on Page 18, which specifically mentions the waiting period for delivery (normal, caesarean section) as 24 months from the first inception of the policy.

Sources used:
[Page 18]
Waiting Period for Delivery - Normal 
Delivery and Caesarean Section
24 months 
for ﬁrst 
delivery 
from ﬁrst 
inception of 
the policy
24 months 
for ﬁrst 
delivery 
from ﬁrst 
inception of 
the policy
24 months 
for ﬁrst 
delivery 
from ﬁrst 
inception of 
the policy
24 months 
for ﬁrst 
delivery 
from ﬁrst 
inception of 
the policy
24 months 
for ﬁrst 
delivery 
from ﬁrst 
inception of 
the policy
24 months 
for ﬁrst 
delivery 
from ﬁrst 
inception of 
the policy
24 months 
for ﬁrst

[Page 13]
"Waiting Periods", "Speciﬁc Waiting Periods" in the event of payment of premium 
within the stipulated grace Period.
iv. No interest will be charged lf the instalment premium is not paid on due

In [12]:
import sys
import os
sys.path.append("..")

from dotenv import load_dotenv
load_dotenv("../.env")

from app.services.rag_chain import get_rag_chain

chain = get_rag_chain()

# Test singleton
chain2 = get_rag_chain()
print(f"Same object? {chain is chain2}")  # should print True

# Test a question
result = chain.invoke("What is the waiting period for pre-existing diseases?")
print(f"\nAnswer:\n{result['answer']}")
print(f"\nSources:\n{result['sources']}")

Same object? True

Answer:
The waiting period for pre-existing diseases is 24 months from the date of inception of the policy (Page 11) and also 36 months for coverage under the policy after the expiry for any pre-existing disease, subject to the same being declared at the time of application and accepted by Insurer (Page 10). Additionally, pre-existing diseases are defined and have specific conditions on Page 4.

Sources:
[Page 4]
advice of a Medical Practitioner. The Insured is not admitted as a day care or in-patient.
Pre-Existing Disease: Pre-existing Disease means any condition, ailment, injury or 
disease:
a) 
That is/are diagnosed by a physician within 48 months prior to the effective date of the 
policy issued by the insurer or its reinstatement
or
b) 
For which medical advice or treatment was recommended by, or received from, a

[Page 11]
waiting period of 24 months from the date of inception of this policy
Note: The above mentioned waiting period will not apply for treatment 